# Chakra Ñawi · contador de broca en Colab (plan B / reentrenamiento con GPU)

Regenera negativos difíciles **sintéticos**, afina YOLOv8n desde el checkpoint v0 y exporta ONNX (fp32 + int8) según `contracts/modelo-io.md`. El modelo nuevo se llama `broca-y8n-v0-sint-dificiles`; `broca-y8n-v1` queda reservado para el sustituto con fotos reales.

1. Entorno de ejecución → Cambiar tipo → **GPU T4**.
2. Sube `ml.zip` con los scripts y `runs/y8n_sint_v0/weights/best.pt`. Excluye `.venv`, `datasets` y el resto de `runs`. El checkpoint es necesario para afinar desde v0.
3. Para **fotos reales** de gorgojos, sigue los pasos del README: rectificar, etiquetar, separar por sesión y mezclar reales y sintéticos. Estas celdas no preparan ni evalúan datos reales; el reporte futuro debe decir **sustituto**.
4. Este notebook usa 60 épocas en GPU; no equivale al presupuesto de 45 minutos de CPU del experimento P2. No comparar los resultados como si fueran la misma corrida. El entorno Windows conserva torch 2.5.1+cpu; Colab usa su entorno CUDA.

**Lo que entrena aquí con datos sintéticos NO es desempeño en broca real.**

In [ ]:
import os
if not os.path.exists('ml'):
    from google.colab import files
    up = files.upload()  # sube ml.zip
    !unzip -q ml.zip -d .
%cd ml
!pip -q install ultralytics onnx onnxruntime onnxslim opencv-python-headless
!nvidia-smi -L

In [ ]:
# Datos sintéticos (marcos 1216×1216 → mosaicos 640×640). En Colab hay 2 CPU: tarda ~20 min con 400/60/60.
!python sintetico.py --salida datasets/sint_dificiles_v1 --semilla 20261004 --train 400 --val 60 --test 60 --workers 2

In [ ]:
# Solo SINTÉTICOS. Requiere una sesión GPU y el checkpoint v0 subido.
import torch
import json, time
from pathlib import Path
assert torch.cuda.is_available(), 'Selecciona una sesión GPU T4 en Colab'
assert Path('runs/y8n_sint_v0/weights/best.pt').exists(), 'Incluye el checkpoint v0 en ml.zip'
from ultralytics import YOLO
m = YOLO('runs/y8n_sint_v0/weights/best.pt')
inicio = time.monotonic()
res = m.train(data='datasets/sint_dificiles_v1/data.yaml', imgsz=640, epochs=60, batch=32, device=0, workers=2,
        project='runs', name='y8n_sint_dificiles_gpu', exist_ok=True, single_cls=True, cache='ram', plots=False,
        seed=0, freeze=4, nbs=32, warmup_epochs=0.3,
        close_mosaic=5, mosaic=1.0, scale=0.15, translate=0.1, degrees=0.0, fliplr=0.5, flipud=0.5,
        hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, max_det=1000)
run = Path('runs/y8n_sint_dificiles_gpu')
resumen = {'sintetico': True, 'nota': 'SOLO SINTÉTICOS; corrida GPU diferente de CPU',
           'modelo_inicial': 'runs/y8n_sint_v0/weights/best.pt', 'datos': 'datasets/sint_dificiles_v1/data.yaml',
           'epocas_completadas': len((run/'results.csv').read_text().strip().splitlines())-1,
           'duracion_s': round(time.monotonic()-inicio, 1), 'device': 0}
(run/'resumen_entrenamiento.json').write_text(json.dumps(resumen, ensure_ascii=False, indent=1), encoding='utf-8')

In [ ]:
# E/S congelada. Modelo GPU separado de la corrida CPU y sin activar reglas.
!python exportar.py --pesos runs/y8n_sint_dificiles_gpu/weights/best.pt --nombre broca-y8n-v0-sint-dificiles-gpu --no-copiar
!python evaluar.py --nombre broca-y8n-v0-sint-dificiles-gpu --marcos datasets/sint_dificiles_v1/marcos/test --val datasets/sint_dificiles_v1/marcos/val --salida resultados/gpu_dificiles
!python contador_clasico.py --calibrar --val datasets/sint_dificiles_v1/marcos/val --params contador_clasico_dificiles_gpu.json
!python contador_clasico.py --test datasets/sint_dificiles_v1/marcos/test --params contador_clasico_dificiles_gpu.json --evaluacion resultados/gpu_dificiles/eval_sintetico.json --salida resultados/gpu_dificiles

In [ ]:
# Descarga los modelos y resultados; copia los .onnx a apps/campo/public/models/ en la PC
!zip -qr salida_ml.zip modelos resultados contador_clasico_dificiles_gpu.json runs/y8n_sint_dificiles_gpu/weights/best.pt runs/y8n_sint_dificiles_gpu/resumen_entrenamiento.json
from google.colab import files
files.download('salida_ml.zip')